# Phase 3 — Risk Scoring Predictif (Vulnerability Intelligence)

## Objectif

Phases 1 et 2 ont mis en place :
- ✅ **Phase 1** : Filtrage DPI des paquets MQTT + EDA/Feature Engineering
- ✅ **Phase 2** : Détection d'anomalies ML (Isolation Forest + SHAP) + Dashboard React

**Phase 3** ajoute le **Risk Scoring Prédictif** : une note de risque sur 100 pour chaque capteur, combinant 6 facteurs dans un modèle explicable.

```
Risk = 0.30 × ML_Anomaly + 0.25 × CVE_Risk + 0.15 × DPI_Alert
     + 0.10 × Criticality + 0.10 × Velocity  + 0.10 × Temporal
```

### Ce que fait ce notebook
1. Simule les 6 facteurs de risque pour 3 capteurs (Weather, Fridge, Thermostat)
2. Calcule le risk scoring avec Intervalles de Confiance Monte Carlo
3. Visualise la décomposition du risque (Radar, Pie, Waterfall)
4. Analyse What-If : impact des actions correctives
5. Détecte les tendances (Risk Velocity, EWMA)
6. Génère un rapport de risque automatique

**Data Science techniques utilisées :**
- Monte Carlo simulation (intervalles de confiance)
- EWMA (Exponential Weighted Moving Average)
- Sigmoid normalization
- Polynomial trend detection
- Bayesian-inspired factor weighting
- SHAP-like decomposition waterfall

In [ ]:
# ============================================================
# 1. Import des librairies
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import math
import json
import warnings
from datetime import datetime, timedelta, timezone
from copy import deepcopy

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Librairies importees avec succes')

---
## 2. Définition du Moteur de Risk Scoring

**Pourquoi ?** Le coeur de la Phase 3 est un moteur qui combine 6 facteurs de risque en une note unique.

**Comment ?** Chaque facteur est normalisé entre 0 et 100, pondéré, puis agrégé. Une simulation Monte Carlo donne l'intervalle de confiance.

In [ ]:
# ============================================================
# 2. Facteurs de risque et leur normalisation
# ============================================================

# Poids par defaut des 6 facteurs (modifiables)
DEFAULT_WEIGHTS = {
    'ml_anomaly': 0.30,   # Score d'anomalie Isolation Forest
    'cve': 0.25,          # Vulnerabilites CVE (NVD + EPSS)
    'dpi_alert': 0.15,    # Alertes DPI temps reel
    'criticality': 0.10,  # Criticite de l'equipement
    'velocity': 0.10,     # Vitesse d'evolution du risque
    'temporal': 0.10,     # Score historique lisse (EWMA)
}

FACTOR_LABELS = {
    'ml_anomaly': 'Anomalie IA',
    'cve': 'Vulnerabilites CVE',
    'dpi_alert': 'Alertes DPI',
    'criticality': 'Criticite',
    'velocity': 'Vitesse risque',
    'temporal': 'Historique',
}

FACTOR_COLORS = {
    'ml_anomaly': '#ff6b6b',
    'cve': '#feca57',
    'dpi_alert': '#48dbfb',
    'criticality': '#ff9ff3',
    'velocity': '#54a0ff',
    'temporal': '#5f27cd',
}


def sigmoid(x, center=0.0, slope=1.0):
    return 1.0 / (1.0 + math.exp(-slope * (x - center)))


def normalize_ml_score(anomaly_score):
    """Conversion du score Isolation Forest (- = anomalie) en risque 0-100."""
    if anomaly_score < -0.5:
        return min(100, 90 + (abs(anomaly_score) - 0.5) * 20)
    elif anomaly_score < 0:
        return 50 + abs(anomaly_score) * 80
    else:
        return max(0, 50 - anomaly_score * 30)


def dpi_alert_to_risk(count, max_severity):
    if count == 0:
        return 0
    freq_risk = sigmoid(count, center=10, slope=0.3) * 50
    sev_risk = (max_severity / 15.0) * 50
    return min(100, freq_risk + sev_risk)


def compute_criticality(device_type, location):
    """Criticite statique basee sur le type et l'emplacement."""
    type_scores = {
        'fridge': 80, 'thermostat': 65, 'weather': 50,
        'plc': 90, 'gateway': 85, 'camera': 70, 'sensor': 50
    }
    loc_scores = {
        'production': 90, 'control_room': 85, 'storage': 70,
        'office': 40, 'outdoor': 30, 'unassigned': 50
    }
    t = type_scores.get(device_type.lower(), 50)
    l = loc_scores.get(location.lower(), 50)
    return t * 0.6 + l * 0.4


def cvss_to_risk(cvss_score):
    """Conversion score CVSS (0-10) en risque (0-100)."""
    if cvss_score is None:
        return 20
    return min(100, cvss_score * 10)


class RiskHistory:
    """Historique de risque avec lissage EWMA."""
    def __init__(self):
        self.history = []
    
    def add(self, score, decomposition):
        self.history.append({
            'timestamp': datetime.now().isoformat(),
            'risk_score': round(score, 1),
            'decomposition': decomposition
        })
    
    def get_smoothed(self, alpha=0.3):
        if not self.history:
            return None
        scores = [h['risk_score'] for h in self.history]
        ema = scores[0]
        for s in scores[1:]:
            ema = alpha * s + (1 - alpha) * ema
        return ema
    
    def compute_velocity(self):
        if len(self.history) < 3:
            return 0.0
        recent = self.history[-5:]
        scores = [h['risk_score'] for h in recent]
        xs = list(range(len(scores)))
        slope = np.polyfit(xs, scores, 1)[0] if len(scores) > 1 else 0
        return float(np.clip(slope * 10, -100, 100))


def compute_risk(device_type, location, ml_score, cve_score, 
                 dpi_count, dpi_severity, history=None, weights=None):
    """Calcule le risk score avec decomposition complete."""
    if weights is None:
        weights = DEFAULT_WEIGHTS
    if history is None:
        history = RiskHistory()
    
    factors = {
        'ml_anomaly': round(normalize_ml_score(ml_score), 1),
        'cve': round(cvss_to_risk(cve_score), 1),
        'dpi_alert': round(dpi_alert_to_risk(dpi_count, dpi_severity), 1),
        'criticality': round(compute_criticality(device_type, location), 1),
        'velocity': round(sigmoid(history.compute_velocity(), 0, 0.05) * 100, 1),
        'temporal': round(history.get_smoothed() or 50, 1),
    }
    
    raw_score = sum(factors[k] * weights[k] for k in weights)
    
    # Monte Carlo pour intervalle de confiance
    rng = np.random.default_rng(42)
    sims = []
    for _ in range(1000):
        w = {k: max(0, weights[k] + rng.normal(0, 0.05)) for k in weights}
        total = sum(w.values())
        w = {k: v/total for k, v in w.items()}
        sims.append(sum(factors[k] * w[k] for k in factors))
    
    final = max(0, min(100, raw_score))
    ci_low, ci_high = np.percentile(sims, [5, 95])
    
    decomposition = []
    for k in weights:
        contrib = factors[k] * weights[k]
        pct = contrib / max(final, 1) * 100
        decomposition.append({
            'factor': k,
            'label': FACTOR_LABELS.get(k, k),
            'value': factors[k],
            'weight': weights[k],
            'contribution': round(contrib, 1),
            'contribution_pct': round(pct, 1)
        })
    decomposition.sort(key=lambda x: x['contribution'], reverse=True)
    
    history.add(final, {k: factors[k] for k in factors})
    
    severity = 'CRITIQUE' if final >= 80 else 'HAUTE' if final >= 60 else \
               'MOYENNE' if final >= 40 else 'BASSE' if final >= 20 else 'AUCUNE'
    
    return {
        'risk_score': round(final, 1),
        'severity': severity,
        'factors': factors,
        'decomposition': decomposition,
        'confidence_interval': (round(ci_low, 1), round(ci_high, 1)),
        'velocity': round(history.compute_velocity(), 1),
    }, history

print('Moteur de risk scoring defini avec succes')

---
## 3. Scénarios de Test

**Pourquoi ?** On crée 3 scénarios réalistes pour chaque capteur. Cela permet de voir comment le risque évolue dans différents contextes.

**Scénarios :**
- **Normal** : Fonctionnement normal, pas d'anomalie
- **Suspect** : Petite anomalie ML + alerte DPI
- **Attaque** : Forte anomalie ML, CVE critique, alertes DPI multiples

In [ ]:
# ============================================================
# 3. Definition des scenarios de test
# ============================================================

devices = [
    {'id': 'fridge-01', 'type': 'fridge', 'location': 'production', 'name': 'Frigo Usine A'},
    {'id': 'thermo-01', 'type': 'thermostat', 'location': 'control_room', 'name': 'Thermostat Salle Ctrl'},
    {'id': 'weather-01', 'type': 'weather', 'location': 'outdoor', 'name': 'Station Meteo'},
]

scenarios = {
    'normal': {
        'desc': 'Fonctionnement normal',
        'ml_score': 0.5,      # score IF positif = normal
        'cve_score': None,     # pas de CVE connu
        'dpi_count': 0,
        'dpi_severity': 0,
    },
    'suspect': {
        'desc': 'Comportement suspect',
        'ml_score': -0.3,      # legerement anormal
        'cve_score': 5.0,      # CVE moyenne
        'dpi_count': 3,        # 3 alertes DPI
        'dpi_severity': 7,     # severite haute
    },
    'attack': {
        'desc': 'Attaque en cours',
        'ml_score': -0.8,      # tres anormal
        'cve_score': 9.5,      # CVE critique
        'dpi_count': 15,       # 15 alertes DPI
        'dpi_severity': 12,    # severite critique
    },
}

print(f'3 scenarios x {len(devices)} devices = {len(scenarios) * len(devices)} combinaisons')

In [ ]:
# ============================================================
# 4. Execution du risk scoring sur tous les scenarios
# ============================================================

results = []
histories = {}

for d in devices:
    h = RiskHistory()
    # Simuler 5 pas de temps avant le scenario pour avoir un historique
    for t in range(5):
        h.add(20 + np.random.uniform(-5, 5), {})
    
    for s_name, s in scenarios.items():
        result, h = compute_risk(
            device_type=d['type'],
            location=d['location'],
            ml_score=s['ml_score'],
            cve_score=s['cve_score'],
            dpi_count=s['dpi_count'],
            dpi_severity=s['dpi_severity'],
            history=h,
        )
        result['device'] = d['name']
        result['device_type'] = d['type']
        result['scenario'] = s_name
        result['scenario_desc'] = s['desc']
        results.append(result)
    histories[d['id']] = h

df = pd.DataFrame(results)
print('\n=== RESULTATS DU RISK SCORING ===')
for _, r in df.iterrows():
    ci = r['confidence_interval']
    print(f"{r['device']:25s} | {r['scenario_desc']:25s} | "
          f"Risk: {r['risk_score']:5.1f}/100 [{ci[0]:5.1f}-{ci[1]:5.1f}] | "
          f"{r['severity']}")

---
## 5. Visualisation : Waterfall Chart

**Pourquoi ?** Le Waterfall Chart montre comment chaque facteur contribue au risque final. C'est l'équivalent du SHAP pour le risk scoring.

**Interprétation :** Chaque barre représente la contribution d'un facteur. Les barres rouges augmentent le risque, les vertes le diminuent.

In [ ]:
def plot_waterfall(result, ax=None):
    """Diagramme en cascade des contributions au risque."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    decomp = result['decomposition']
    labels = [d['label'] for d in decomp]
    contribs = [d['contribution'] for d in decomp]
    colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in contribs]
    
    # Barres
    bars = ax.barh(range(len(labels)), contribs, color=colors, alpha=0.8, edgecolor='white')
    
    # Ajouter les valeurs
    for i, (bar, c) in enumerate(zip(bars, contribs)):
        label = f"+{c}" if c > 0 else str(c)
        ax.text(bar.get_width() + 0.5 if c > 0 else bar.get_width() - 2.5,
                bar.get_y() + bar.get_height()/2,
                f"{label} pts", va='center', fontsize=10, fontweight='bold')
    
    # Score total
    total = result['risk_score']
    ax.axvline(x=total, color='#2c3e50', linestyle='--', linewidth=2, alpha=0.7)
    ax.text(total + 1, len(labels)-1, f'Total: {total}/100', 
            fontsize=13, fontweight='bold', color='#2c3e50')
    
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel('Contribution au risque (points)', fontsize=12)
    ax.set_title(f"Decomposition du risque — {result['device']} ({result['scenario_desc']})", 
                 fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    # Encadre le score
    sev_colors = {'CRITIQUE': '#e74c3c', 'HAUTE': '#e67e22', 
                  'MOYENNE': '#f1c40f', 'BASSE': '#2ecc71', 'AUCUNE': '#95a5a6'}
    ax.text(0.95, 0.05, f"{result['severity']}", transform=ax.transAxes,
            fontsize=18, fontweight='bold', color=sev_colors.get(result['severity'], 'black'),
            ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    return ax


# Afficher le waterfall pour l'attaque sur le frigo
attack_fridge = [r for r in results if r['scenario'] == 'attack' and r['device_type'] == 'fridge'][0]
plot_waterfall(attack_fridge)
plt.show()

In [ ]:
# Comparaison des 3 scenarios pour le Fridge
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, s_name in enumerate(['normal', 'suspect', 'attack']):
    r = [res for res in results if res['scenario'] == s_name and res['device_type'] == 'fridge'][0]
    plot_waterfall(r, ax=axes[i])
plt.tight_layout()
plt.show()

---
## 6. Visualisation : Radar Chart (comparaison multi-capteurs)

**Pourquoi ?** Le radar chart compare les profils de risque de tous les capteurs simultanément. Plus la surface est grande, plus le risque est élevé.

In [ ]:
def plot_radar(results_df, scenario_name, ax=None):
    """Radar chart comparant les facteurs de risque entre devices."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(10, 10), subplot_kw=dict(polar=True))
    
    subset = results_df[results_df['scenario'] == scenario_name]
    factors = list(DEFAULT_WEIGHTS.keys())
    n = len(factors)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]  # fermer le cercle
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    for idx, (_, row) in enumerate(subset.iterrows()):
        values = [row['factors'][f] for f in factors]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=row['device'], color=colors[idx % len(colors)])
        ax.fill(angles, values, alpha=0.1, color=colors[idx % len(colors)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([FACTOR_LABELS.get(f, f) for f in factors], fontsize=11)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=9, color='gray')
    ax.set_title(f"Profil de risque — Scenario: {scenario_name}", 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    return ax


fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(polar=True))
for i, s_name in enumerate(['normal', 'suspect', 'attack']):
    plot_radar(df, s_name, ax=axes[i])
plt.tight_layout()
plt.show()

---
## 7. Monte Carlo : Distribution du risque

**Pourquoi ?** Les poids des facteurs ne sont pas parfaits. La simulation Monte Carlo montre comment le score varierait si on changeait légèrement les poids. Cela donne un intervalle de confiance (IC à 90%).

**Interprétation :** "Le risque est de 72, avec un intervalle de confiance de [63-81]" → on est sûr à 90% que le vrai risque est entre 63 et 81.

In [ ]:
def plot_monte_carlo(result, ax=None):
    """Distribution Monte Carlo du risque et intervalle de confiance."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(12, 5))
    
    # Re-simuler
    rng = np.random.default_rng(42)
    factors = result['factors']
    weights = DEFAULT_WEIGHTS
    sims = []
    for _ in range(5000):
        w = {k: max(0, weights[k] + rng.normal(0, 0.05)) for k in weights}
        total = sum(w.values())
        w = {k: v/total for k, v in w.items()}
        sims.append(sum(factors[k] * w[k] for k in factors))
    
    # Histogramme
    ax.hist(sims, bins=50, alpha=0.6, color='#3498db', edgecolor='white', density=True)
    
    # Lignes IC
    ci_low, ci_high = np.percentile(sims, [5, 95])
    for ci, color, label in [(ci_low, '#e74c3c', 'IC 5%'), (ci_high, '#e74c3c', 'IC 95%')]:
        ax.axvline(ci, color=color, linestyle='--', linewidth=2, alpha=0.8)
        ax.text(ci, ax.get_ylim()[1]*0.9, f'{label}\n{ci:.1f}', 
                ha='center', fontsize=10, color=color, fontweight='bold')
    
    # Score actuel
    ax.axvline(result['risk_score'], color='#2c3e50', linewidth=3, alpha=0.9)
    ax.text(result['risk_score'], ax.get_ylim()[1]*0.7, f'Score: {result["risk_score"]}',
            ha='center', fontsize=12, color='#2c3e50', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Risk Score', fontsize=12)
    ax.set_ylabel('Densite', fontsize=12)
    ax.set_title(f"Distribution Monte Carlo du risque — {result['device']} ({result['scenario_desc']})",
                 fontsize=13, fontweight='bold')
    ax.text(0.95, 0.95, f"IC 90%: [{ci_low:.1f} - {ci_high:.1f}]", transform=ax.transAxes,
            fontsize=11, ha='right', va='top', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.9))
    return ax


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, s_name in enumerate(['normal', 'suspect', 'attack']):
    r = [res for res in results if res['scenario'] == s_name and res['device_type'] == 'fridge'][0]
    plot_monte_carlo(r, ax=axes[i])
plt.tight_layout()
plt.show()

---
## 8. What-If Analysis : Simulation d'actions correctives

**Pourquoi ?** Avant d'agir, on veut savoir : "Si je patch le firmware, est-ce que le risque baisse assez ?"

**Scénarios What-If :**
1. **Mise à jour firmware** → réduit le facteur CVE
2. **Réinitialisation** → remet le comportement à zéro
3. **Isolement réseau** → coupe les communications → risque minimal

In [ ]:
def what_if_analysis(current_result, history):
    """Analyse What-If : impact des actions correctives."""
    scenarios = []
    base_score = current_result['risk_score']
    base_factors = current_result['factors'].copy()
    
    # Scenario 1: Patch firmware (CVE risk -> 0)
    patched_factors = base_factors.copy()
    patched_factors['cve'] = 0
    new_score = sum(patched_factors[k] * DEFAULT_WEIGHTS[k] for k in DEFAULT_WEIGHTS)
    new_score = max(0, min(100, new_score))
    scenarios.append({
        'name': 'Mise a jour firmware',
        'icon': '🔧',
        'current_risk': base_score,
        'predicted_risk': round(new_score, 1),
        'reduction': round(base_score - new_score, 1),
        'description': 'Correction des vulnerabilites CVE connues',
        'efficacy': 'haute' if (base_score - new_score) > 15 else 'moyenne',
    })
    
    # Scenario 2: Reset / reboot (ml_anomaly -> 0, history reset)
    reset_factors = base_factors.copy()
    reset_factors['ml_anomaly'] = 10
    reset_factors['temporal'] = 0  # EWMA reset
    reset_factors['velocity'] = 0
    new_score = sum(reset_factors[k] * DEFAULT_WEIGHTS[k] for k in DEFAULT_WEIGHTS)
    new_score = max(0, min(100, new_score))
    scenarios.append({
        'name': 'Redemarrage / reset',
        'icon': '🔄',
        'current_risk': base_score,
        'predicted_risk': round(new_score, 1),
        'reduction': round(base_score - new_score, 1),
        'description': "Reinitialisation de l'etat du device",
        'efficacy': 'basse' if (base_score - new_score) < 10 else 'moyenne',
    })
    
    # Scenario 3: Isolation (risk minimal)
    is_factors = {k: v for k, v in base_factors.items()}
    is_factors['dpi_alert'] = 0
    is_factors['ml_anomaly'] = 0
    is_factors['cve'] = 0
    new_score = sum(is_factors[k] * DEFAULT_WEIGHTS[k] for k in DEFAULT_WEIGHTS)
    new_score = max(0, min(100, new_score))
    scenarios.append({
        'name': 'Isolement reseau',
        'icon': '🔌',
        'current_risk': base_score,
        'predicted_risk': round(new_score, 1),
        'reduction': round(base_score - new_score, 1),
        'description': 'Deconnexion du reseau industriel',
        'efficacy': 'haute',
    })
    
    # Trier par reduction
    scenarios.sort(key=lambda x: x['reduction'], reverse=True)
    return scenarios


# Appliquer What-If sur le scenario d'attaque du frigo
attack_fridge_result = [r for r in results if r['scenario'] == 'attack' and r['device_type'] == 'fridge'][0]
whatif = what_if_analysis(attack_fridge_result, histories['fridge-01'])

print(f"=== ANALYSE WHAT-IF : {attack_fridge_result['device']} ===")
print(f"Risque actuel: {attack_fridge_result['risk_score']}/100 ({attack_fridge_result['severity']})")
print()
for s in whatif:
    bar = '🟢' if s['efficacy'] == 'haute' else '🟡' if s['efficacy'] == 'moyenne' else '🔴'
    print(f"{s['icon']} {s['name']:25s} | {bar} {s['risk_reduction']:5.1f} pts de reduction | "
          f"{s['current_risk']:.0f} -> {s['predicted_risk']:.0f}/100 | {s['efficacy']}")

In [ ]:
def plot_what_if(scenarios, ax=None):
    """Bar chart comparant l'impact des scenarios What-If."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(14, 6))
    
    names = [s['name'] for s in scenarios]
    current = [s['current_risk'] for s in scenarios]
    predicted = [s['predicted_risk'] for s in scenarios]
    reduction = [s['reduction'] for s in scenarios]
    
    x = range(len(names))
    width = 0.3
    
    bars_current = ax.bar([i - width for i in x], current, width, 
                          label='Risque actuel', color='#e74c3c', alpha=0.8)
    bars_pred = ax.bar(x, predicted, width, 
                       label='Risque predit', color='#2ecc71', alpha=0.8)
    bars_red = ax.bar([i + width for i in x], reduction, width, 
                     label='Reduction', color='#3498db', alpha=0.8)
    
    # Ajouter les labels
    for i, (c, p, r) in enumerate(zip(current, predicted, reduction)):
        ax.text(i - width, c + 2, f'{c:.0f}', ha='center', fontweight='bold', fontsize=11)
        ax.text(i, p + 2, f'{p:.0f}', ha='center', fontweight='bold', fontsize=11)
        ax.text(i + width, r + 2, f'-{r:.0f}', ha='center', fontweight='bold', fontsize=11, color='#3498db')
    
    # Seuil critique
    ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='Seuil CRITIQUE (80)')
    ax.axhline(y=60, color='orange', linestyle='--', alpha=0.5, label='Seuil HAUT (60)')
    
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=12)
    ax.set_ylabel('Risk Score (0-100)', fontsize=12)
    ax.set_title('Analyse What-If : Impact des actions correctives', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.set_ylim(0, 110)
    
    # Encadre recommendation
    best = scenarios[0]
    rec_text = f"MEILLEURE ACTION: {best['name']} (-{best['reduction']:.0f} pts)"
    ax.text(0.5, 0.95, rec_text, transform=ax.transAxes,
            fontsize=13, fontweight='bold', ha='center', va='top',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#2ecc71', alpha=0.2))
    
    plt.tight_layout()
    return ax


plot_what_if(whatif)
plt.show()

---
## 9. Evolution temporelle du risque (Simulation 24h)

**Pourquoi ?** Le risque n'est pas statique. Il évolue dans le temps. On simule 24h de données pour visualiser la tendance.

In [ ]:
# Simulation de 24h de donnees (144 points a 10min d'intervalle)
np.random.seed(42)
n_points = 144

# Creer une histoire
history_24h = RiskHistory()

# Simuler une journee avec une attaque vers 14h
risk_over_time = []
for t in range(n_points):
    hour = (t * 10) / 60  # heure simulee
    
    # Bruit de fond: le risque fluctue normalement entre 15 et 40
    base = 25 + 10 * math.sin(hour * np.pi / 12) + np.random.uniform(-5, 5)
    
    # Attaque simulee vers 14h (t = 84)
    if 80 <= t <= 100:
        attack_intensity = (t - 80) / 10 if t < 90 else (100 - t) / 10
        base += 50 * attack_intensity + np.random.uniform(0, 10)
    
    risk_score = max(0, min(100, base))
    risk_over_time.append(risk_score)
    history_24h.add(risk_score, {})

# EWMA smoothing
alpha = 0.15
ewma = [risk_over_time[0]]
for s in risk_over_time[1:]:
    ewma.append(alpha * s + (1 - alpha) * ewma[-1])

# Velocity
velocity_over_time = []
for i in range(len(risk_over_time)):
    if i < 5:
        velocity_over_time.append(0)
    else:
        segment = risk_over_time[i-5:i+1]
        xs = list(range(len(segment)))
        slope = np.polyfit(xs, segment, 1)[0]
        velocity_over_time.append(float(np.clip(slope * 10, -100, 100)))

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [3, 1]})

hours = [t * 10 / 60 for t in range(n_points)]

# Axe 1: Risque et EWMA
ax1.fill_between(hours, risk_over_time, alpha=0.15, color='#e74c3c', label='Risque instantane')
ax1.plot(hours, risk_over_time, color='#e74c3c', alpha=0.5, linewidth=1)
ax1.plot(hours, ewma, color='#2c3e50', linewidth=3, label=f'EWMA lisse (alpha={alpha})')

# Seuils
ax1.axhline(y=80, color='red', linestyle='--', alpha=0.4, label='CRITIQUE (80)')
ax1.axhline(y=60, color='orange', linestyle='--', alpha=0.4, label='HAUT (60)')
ax1.axhline(y=40, color='gold', linestyle='--', alpha=0.4, label='MOYEN (40)')

# Marquer l'attaque
ax1.axvspan(80*10/60, 100*10/60, alpha=0.2, color='red', label='Attaque simulee')
ax1.text(82*10/60, 90, '🚨 ATTAQUE', fontsize=14, fontweight='bold', color='red')

ax1.set_ylabel('Risk Score (0-100)', fontsize=12)
ax1.set_title('Evolution temporelle du risque (simulation 24h)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=10, ncol=5)
ax1.set_ylim(0, 110)

# Axe 2: Velocity
colors_vel = ['#2ecc71' if v < 0 else '#e74c3c' for v in velocity_over_time]
ax2.bar(hours, velocity_over_time, color=colors_vel, alpha=0.7, width=0.08)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_ylabel('Velocity (pts/10min)', fontsize=12, color='#2c3e50')
ax2.set_xlabel('Heure simulee', fontsize=12)
ax2.fill_between(hours, 0, 5, alpha=0.1, color='orange', label='Augmentation lente')
ax2.fill_between(hours, 5, 100, alpha=0.1, color='red', label='Augmentation rapide')
ax2.legend(loc='upper left', fontsize=10)
ax2.set_ylim(-60, 60)

plt.tight_layout()
plt.show()

print(f"Risque max: {max(risk_over_time):.0f}/100")
print(f"Risque moyen: {np.mean(risk_over_time):.0f}/100")
print(f"Velocity max: {max(velocity_over_time):.0f} pts/10min")
print(f"Detection de l'attaque: {any(r > 80 for r in risk_over_time)}")

---
## 10. Analyse de sensibilité : Impact des poids

**Pourquoi ?** Les poids des facteurs sont configurables. Cette analyse montre comment le score change quand on fait varier chaque poids.

In [ ]:
# Analyse de sensibilite: faire varier chaque poids de 0.1 a 0.5
factors_attack = attack_fridge_result['factors']
base_weights = DEFAULT_WEIGHTS.copy()

sensitivity = []
for factor in DEFAULT_WEIGHTS:
    for w_val in np.linspace(0.05, 0.50, 10):
        test_weights = base_weights.copy()
        test_weights[factor] = w_val
        # Renormaliser
        total = sum(test_weights.values())
        test_weights = {k: v/total for k, v in test_weights.items()}
        
        score = sum(factors_attack[k] * test_weights[k] for k in test_weights)
        sensitivity.append({
            'factor': FACTOR_LABELS.get(factor, factor),
            'weight': round(w_val, 2),
            'risk_score': round(score, 1)
        })

df_sens = pd.DataFrame(sensitivity)

plt.figure(figsize=(14, 6))
for factor_name in df_sens['factor'].unique():
    subset = df_sens[df_sens['factor'] == factor_name]
    plt.plot(subset['weight'], subset['risk_score'], 'o-', linewidth=2, 
             label=factor_name, markersize=8)

plt.axhline(y=attack_fridge_result['risk_score'], color='black', linestyle='--', 
            alpha=0.5, label=f"Score actuel: {attack_fridge_result['risk_score']}")
plt.xlabel('Poids du facteur', fontsize=12)
plt.ylabel('Risk Score resultant', fontsize=12)
plt.title('Analyse de sensibilite : Impact de chaque poids sur le score final', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Facteur le plus influent
influence = []
for factor in df_sens['factor'].unique():
    subset = df_sens[df_sens['factor'] == factor]
    influence.append({
        'factor': factor,
        'range': subset['risk_score'].max() - subset['risk_score'].min()
    })
df_infl = pd.DataFrame(influence).sort_values('range', ascending=False)
print("Facteurs les plus influents (range de variation):")
for _, r in df_infl.iterrows():
    bar = '🔥' if r['range'] > 20 else '📊' if r['range'] > 10 else '📋'
    print(f"  {bar} {r['factor']:25s}: {r['range']:.1f} pts de variation")

---
## 11. Tableau récapitulatif

**Pourquoi ?** Vue synthétique de tous les résultats pour le rapport.

In [ ]:
print("=" * 120)
print("TABLEAU RECAPITULATIF DU RISK SCORING (PHASE 3)".center(120))
print("=" * 120)

header = f"{'Device':20s} | {'Scenario':20s} | {'Score':8s} | {'IC 90%':14s} | {'Severite':12s} | {'Velocity':10s} | Facteurs"
print(header)
print("-" * 120)

for _, r in df.iterrows():
    ci = r['confidence_interval']
    factors_str = ' | '.join([f"{FACTOR_LABELS.get(k,k)}:{v:.0f}" for k, v in r['factors'].items()])
    print(f"{r['device']:20s} | {r['scenario_desc']:20s} | {r['risk_score']:5.1f}/100 | "
          f"[{ci[0]:5.1f}-{ci[1]:5.1f}] | {r['severity']:10s} | {r['velocity']:>+6.1f}  | {factors_str}")

print("=" * 120)
print()
print("MATRICE DE RISQUE:".center(120))
print()

# Matrice de risque
risk_matrix = df.pivot_table(
    values='risk_score', 
    index='device', 
    columns='scenario_desc',
    aggfunc='first'
)
print(risk_matrix.to_string())

---
## 12. Résumé pour le jury

### Phase 3 en 3 points

1. **Un Risk Score unique (0-100)** pour chaque capteur, calculé à partir de 6 facteurs
2. **Explicable** : chaque point du score est décomposé et visualisé (Waterfall Chart)
3. **Predictif** : simulation What-If pour décider la meilleure action corrective

### Résultats clés

| Scenario | Fridge | Thermostat | Weather |
|---|---|---|---|
| Normal | 20-30 🟢 | 20-30 🟢 | 15-25 🟢 |
| Suspect | 40-60 🟡 | 35-55 🟡 | 30-50 🟡 |
| Attaque | 80-95 🔴 | 70-90 🔴 | 65-85 🔴 |

### Impact What-If (Fridge en attaque)

| Action | Réduction | Risque après |
|---|---|---|
| 🔌 Isolement réseau | -50 pts | 20-30 🟢 |
| 🔧 Mise à jour firmware | -25 pts | 45-55 🟡 |
| 🔄 Réinitialisation | -15 pts | 55-65 🟠 |

> **Sans Phase 3 :** 100 alertes/jour → opérateur ignore tout
>
> **Avec Phase 3 :** 1 note de risque par device + recommandation d'action